# Lab Tùy chọn: Biểu diễn mô hình

<figure>
 <img src="./images/C1_W1_L3_S1_Lecture_b.png"   style="width:600px;height:200px;">
</figure>

## Mục tiêu
Trong lab này, bạn sẽ:
- Học cách triển khai mô hình $f_{w,b}$ cho hồi quy tuyến tính với một biến

## Ký hiệu
Dưới đây là tóm tắt một số ký hiệu mà bạn sẽ gặp.  

|Ký hiệu <img width=70/> <br />  chung  <img width=70/> | Mô tả<img width=350/>| Python (nếu có) |
|: ------------|: ------------------------------------------------------------||
| $a$ | vô hướng, không in đậm                                                      ||
| $\mathbf{a}$ | vector, in đậm                                                      ||
| **Hồi quy** |         |    |     |
|  $\mathbf{x}$ | Giá trị đặc trưng của ví dụ huấn luyện (trong lab này - Diện tích (1000 sqft))  | `x_train` |   
|  $\mathbf{y}$  | Giá trị mục tiêu của ví dụ huấn luyện (trong lab này Giá (nghìn đô la))  | `y_train` 
|  $x^{(i)}$, $y^{(i)}$ | Ví dụ huấn luyện thứ $i$ | `x_i`, `y_i`|
| m | Số lượng ví dụ huấn luyện | `m`|
|  $w$  |  tham số: trọng số                                 | `w`    |
|  $b$           |  tham số: độ chệch (bias)                                           | `b`    |     
| $f_{w,b}(x^{(i)})$ | Kết quả đánh giá mô hình tại $x^{(i)}$ với tham số $w,b$: $f_{w,b}(x^{(i)}) = wx^{(i)}+b$  | `f_wb` | 


## Công cụ
Trong lab này, bạn sẽ sử dụng: 
- NumPy, một thư viện phổ biến cho tính toán khoa học
- Matplotlib, một thư viện phổ biến để vẽ đồ thị dữ liệu

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

# Phát biểu bài toán
<img align="left" src="./images/C1_W1_L3_S1_trainingdata.png"    style=" width:380px; padding: 10px;  " /> 

Giống như trong bài giảng, bạn sẽ sử dụng ví dụ minh họa về dự đoán giá nhà.  
Lab này sẽ sử dụng một tập dữ liệu đơn giản chỉ với hai điểm dữ liệu - một căn nhà 1000 feet vuông (sqft) được bán với giá \\$300.000 và một căn nhà 2000 feet vuông được bán với giá \\$500.000. Hai điểm này sẽ tạo thành *dữ liệu hay tập huấn luyện* của chúng ta. Trong lab này, đơn vị của diện tích là 1000 sqft và đơn vị của giá là nghìn đô la.

| Diện tích (1000 sqft)     | Giá (nghìn đô la) |
| -------------------| ------------------------ |
| 1.0               | 300                      |
| 2.0               | 500                      |

Bạn muốn khớp một mô hình hồi quy tuyến tính (được thể hiện ở trên bằng đường thẳng màu xanh dương) đi qua hai điểm này, để sau đó có thể dự đoán giá cho các căn nhà khác - ví dụ, một căn nhà 1200 sqft.


Vui lòng chạy ô lệnh sau để tạo các biến `x_train` và `y_train`. Dữ liệu được lưu trong các mảng NumPy một chiều.

In [ ]:
# x_train là biến đầu vào (diện tích tính theo 1000 feet vuông)
# y_train là giá trị mục tiêu (giá tính theo nghìn đô la)
x_train = np.array([1.0, 2.0])
y_train = np.array([300.0, 500.0])
print(f"x_train = {x_train}")
print(f"y_train = {y_train}")

>**Lưu ý**: Khóa học sẽ thường xuyên sử dụng cách định dạng đầu ra 'f-string' của Python được mô tả [tại đây](https://docs.python.org/3/tutorial/inputoutput.html) khi in kết quả. Nội dung nằm giữa các dấu ngoặc nhọn sẽ được tính toán khi tạo ra kết quả in ra.

### Số lượng ví dụ huấn luyện `m`
Bạn sẽ sử dụng `m` để biểu thị số lượng ví dụ huấn luyện. Các mảng Numpy có tham số `.shape`. `x_train.shape` trả về một tuple của Python với một mục cho mỗi chiều. `x_train.shape[0]` là độ dài của mảng và số lượng ví dụ như được minh họa bên dưới.

In [ ]:
# m là số lượng ví dụ huấn luyện
print(f"x_train.shape: {x_train.shape}")
m = x_train.shape[0]
print(f"Number of training examples is: {m}")

Bạn cũng có thể sử dụng hàm `len()` của Python như được minh họa bên dưới.

In [ ]:
# m là số lượng ví dụ huấn luyện
m = len(x_train)
print(f"Number of training examples is: {m}")

### Ví dụ huấn luyện `x_i, y_i`

Bạn sẽ sử dụng (x$^{(i)}$, y$^{(i)}$) để biểu thị ví dụ huấn luyện thứ $i$. Vì Python đánh chỉ số bắt đầu từ 0, (x$^{(0)}$, y$^{(0)}$) là (1.0, 300.0) và (x$^{(1)}$, y$^{(1)}$) là (2.0, 500.0). 

Để truy cập một giá trị trong mảng Numpy, ta đánh chỉ số vào mảng với vị trí (offset) mong muốn. Ví dụ, cú pháp để truy cập vị trí số 0 của `x_train` là `x_train[0]`.
Hãy chạy khối mã tiếp theo bên dưới để lấy ví dụ huấn luyện thứ $i$.

In [ ]:
i = 0 # Đổi thành 1 để xem (x^1, y^1)

x_i = x_train[i]
y_i = y_train[i]
print(f"(x^({i}), y^({i})) = ({x_i}, {y_i})")

### Vẽ đồ thị dữ liệu

Bạn có thể vẽ hai điểm này bằng hàm `scatter()` trong thư viện `matplotlib`, như được minh họa trong ô lệnh bên dưới. 
- Các tham số hàm `marker` và `c` hiển thị các điểm dưới dạng dấu X màu đỏ (mặc định là các chấm tròn màu xanh dương).

Bạn có thể sử dụng các hàm khác trong thư viện `matplotlib` để đặt tiêu đề và các nhãn cần hiển thị

In [ ]:
# Vẽ các điểm dữ liệu
plt.scatter(x_train, y_train, marker='x', c='r')
# Đặt tiêu đề
plt.title("Housing Prices")
# Đặt nhãn trục y
plt.ylabel('Price (in 1000s of dollars)')
# Đặt nhãn trục x
plt.xlabel('Size (1000 sqft)')
plt.show()

## Hàm mô hình

<img align="left" src="./images/C1_W1_L3_S1_model.png"     style=" width:380px; padding: 10px; " > Như đã mô tả trong bài giảng, hàm mô hình cho hồi quy tuyến tính (là hàm ánh xạ từ `x` đến `y`) được biểu diễn như sau 

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$

Công thức trên là cách bạn có thể biểu diễn các đường thẳng - các giá trị khác nhau của $w$ và $b$ cho bạn các đường thẳng khác nhau trên đồ thị. <br/> <br/> <br/> <br/> <br/> 

Hãy thử hiểu rõ hơn về điều này thông qua các khối mã bên dưới. Hãy bắt đầu với $w = 100$ và $b = 100$. 

**Lưu ý: Bạn có thể quay lại ô lệnh này để điều chỉnh các tham số w và b của mô hình**

In [ ]:
w = 100
b = 100
print(f"w: {w}")
print(f"b: {b}")

Bây giờ, hãy tính giá trị của $f_{w,b}(x^{(i)})$ cho hai điểm dữ liệu của bạn. Bạn có thể viết rõ ra cho từng điểm dữ liệu như sau - 

với $x^{(0)}$, `f_wb = w * x[0] + b`

với $x^{(1)}$, `f_wb = w * x[1] + b`

Với một số lượng lớn điểm dữ liệu, cách này có thể trở nên cồng kềnh và lặp lại. Vì vậy, thay vào đó, bạn có thể tính giá trị đầu ra của hàm trong một vòng lặp `for` như được minh họa trong hàm `compute_model_output` bên dưới.
> **Lưu ý**: Mô tả tham số `(ndarray (m,))` mô tả một mảng n chiều của Numpy có shape (m,). `(scalar)` mô tả một tham số không có chiều, chỉ là một độ lớn.  
> **Lưu ý**: `np.zero(n)` sẽ trả về một mảng numpy một chiều với $n$ phần tử   


In [ ]:
def compute_model_output(x, w, b):
    """
    Computes the prediction of a linear model
    Args:
      x (ndarray (m,)): Data, m examples 
      w,b (scalar)    : model parameters  
    Returns
      f_wb (ndarray (m,)): model prediction
    """
    m = x.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = w * x[i] + b
        
    return f_wb

Bây giờ hãy gọi hàm `compute_model_output` và vẽ đồ thị kết quả đầu ra..

In [ ]:
tmp_f_wb = compute_model_output(x_train, w, b,)

# Vẽ dự đoán của mô hình
plt.plot(x_train, tmp_f_wb, c='b',label='Our Prediction')

# Vẽ các điểm dữ liệu
plt.scatter(x_train, y_train, marker='x', c='r',label='Actual Values')

# Đặt tiêu đề
plt.title("Housing Prices")
# Đặt nhãn trục y
plt.ylabel('Price (in 1000s of dollars)')
# Đặt nhãn trục x
plt.xlabel('Size (1000 sqft)')
plt.legend()
plt.show()

Như bạn có thể thấy, việc đặt $w = 100$ và $b = 100$ *không* cho ra một đường thẳng khớp với dữ liệu của chúng ta. 

### Thử thách
Hãy thử nghiệm với các giá trị khác nhau của $w$ và $b$. Giá trị nên là bao nhiêu để có một đường thẳng khớp với dữ liệu của chúng ta?

#### Mẹo:
Bạn có thể dùng chuột nhấp vào phần "Gợi ý" màu xanh lá bên dưới để xem một số gợi ý cho việc chọn b và w.

<details>
<summary>
    <font size='3', color='darkgreen'><b>Gợi ý</b></font>
</summary>
    <p>
    <ul>
        <li>Hãy thử $w = 200$ và $b = 100$ </li>
    </ul>
    </p>

### Dự đoán
Bây giờ chúng ta đã có một mô hình, chúng ta có thể sử dụng nó để thực hiện dự đoán ban đầu của mình. Hãy dự đoán giá của một căn nhà 1200 sqft. Vì đơn vị của $x$ là 1000 sqft, nên $x$ là 1.2.


In [ ]:
w = 200                         
b = 100    
x_i = 1.2
cost_1200sqft = w * x_i + b    

print(f"${cost_1200sqft:.0f} thousand dollars")

# Chúc mừng!
Trong lab này bạn đã học được:
 - Hồi quy tuyến tính xây dựng một mô hình thiết lập mối quan hệ giữa các đặc trưng (features) và mục tiêu (targets)
     - Trong ví dụ trên, đặc trưng là diện tích nhà và mục tiêu là giá nhà
     - đối với hồi quy tuyến tính đơn giản, mô hình có hai tham số $w$ và $b$ có giá trị được 'khớp' bằng cách sử dụng *dữ liệu huấn luyện*.
     - sau khi các tham số của mô hình đã được xác định, mô hình có thể được sử dụng để thực hiện các dự đoán trên dữ liệu mới.